# Week 9: Raster & Remote Sensing

This notebook covers two types of raster analysis:

**Part A: Satellite Imagery & NDVI**
- Calculate vegetation indices from satellite bands
- Detect vegetation change over time

**Part B: Digital Elevation Models (DEMs)**
- Calculate slope and aspect from elevation data
- Create hillshade visualizations
- Terrain analysis techniques

---

## Data options

| Option | Description |
|--------|-------------|
| **Sample data** | Built-in synthetic data. No download needed! |
| **Your own data** | Real satellite imagery or DEMs you download |

**Recommendation:** Start with sample data to understand the concepts, then try real data.

---

## Step 0: Set up environment

This cell detects whether you're running in Google Colab or local Jupyter,
and installs the required packages.

In [ ]:
# ============================================================
# ENVIRONMENT DETECTION AND PACKAGE INSTALLATION
# ============================================================
# This cell checks where the notebook is running and installs
# the geospatial packages we need.

import sys

# Check if we're in Google Colab by looking for the colab module
# sys.modules is a dictionary of all imported modules
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab")
    print("Installing packages (this takes about 1 minute)...")
    
    # The ! prefix runs shell commands from Python
    # -q flag means "quiet" - less output
    !pip install geopandas rasterio rasterstats -q
    
    print("Done!")
else:
    print("Running in local Jupyter")
    print("Make sure you activated your environment: conda activate intro-gis")

---

## Step 1: Set up folder paths

We use a consistent folder structure:
- `data/raw/` — Original input files (never modify these)
- `data/processed/` — Your analysis outputs

In [ ]:
# ============================================================
# SET UP DATA PATHS
# ============================================================
# pathlib.Path provides a clean way to work with file paths
# that works on both Windows and Mac/Linux

from pathlib import Path

if IN_COLAB:
    # In Colab, we need to "mount" Google Drive to access files
    # This connects your Drive to the Colab virtual machine
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Paths point to your Google Drive
    RAW = Path("/content/drive/MyDrive/intro-gis/week09/data/raw")
    PROCESSED = Path("/content/drive/MyDrive/intro-gis/week09/data/processed")
else:
    # Local paths relative to where the notebook is saved
    RAW = Path("data/raw")
    PROCESSED = Path("data/processed")

# Create folders if they don't exist
# parents=True creates parent folders too
# exist_ok=True means don't error if folder already exists
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"Raw data folder: {RAW}")
print(f"Processed folder: {PROCESSED}")

---

## Step 2: Import libraries

Each library serves a specific purpose:
- **numpy** — Fast array math (rasters are just 2D arrays of numbers)
- **geopandas** — Vector data (points, lines, polygons)
- **rasterio** — Reading/writing raster files (GeoTIFF, etc.)
- **matplotlib** — Creating visualizations

In [ ]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import numpy as np                    # Numerical operations on arrays
import geopandas as gpd               # Vector data (shapefiles, GeoJSON)
import matplotlib.pyplot as plt       # Plotting and visualization
from matplotlib import colors         # Custom color maps
from shapely.geometry import box      # Create rectangle geometries

import rasterio                       # Read/write raster files
from rasterio.transform import from_bounds  # Create geotransforms

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

---

# Part A: Satellite Imagery & NDVI

NDVI (Normalized Difference Vegetation Index) measures vegetation health
using the ratio of near-infrared to red light reflected by plants.

## Step 3: Load or generate satellite data

The notebook checks for local files first. If none are found,
it generates realistic synthetic data for learning purposes.

In [ ]:
# ============================================================
# LOAD OR GENERATE SATELLITE DATA
# ============================================================
# We check if you have real satellite files. If not, we generate
# synthetic data that simulates a vegetation clearing event.

# Define paths to potential local files
local_before = RAW / "sentinel_before.tif"
local_after = RAW / "sentinel_after.tif"

# Check if both files exist using Path.exists()
if local_before.exists() and local_after.exists():
    print("Found local satellite imagery - using your data")
    USE_SAMPLE_DATA = False
else:
    print("No local files found - generating sample data...")
    print("(Add sentinel_before.tif and sentinel_after.tif to data/raw/ to use real imagery)\n")
    USE_SAMPLE_DATA = True
    
    # --------------------------------------------------------
    # GENERATE SYNTHETIC SATELLITE DATA
    # --------------------------------------------------------
    # We create fake but realistic satellite imagery to demonstrate
    # NDVI calculation without requiring large file downloads.
    
    # Set random seed for reproducibility
    # This ensures everyone gets the same "random" results
    np.random.seed(42)
    
    # Image dimensions
    # At 100m resolution, 100x100 pixels = 10km x 10km area
    height, width = 100, 100
    
    # Geographic bounds (Sydney region)
    # These are longitude (x) and latitude (y) coordinates
    minx, miny = 151.0, -33.9   # Southwest corner
    maxx, maxy = 151.1, -33.8   # Northeast corner
    
    # Create a geotransform
    # This tells GIS software how to position the raster on Earth
    # from_bounds() calculates pixel size from bounds and dimensions
    transform = from_bounds(minx, miny, maxx, maxy, width, height)
    
    # --------------------------------------------------------
    # CREATE "BEFORE" IMAGE (healthy vegetation)
    # --------------------------------------------------------
    
    # Start with base vegetation (NDVI around 0.6 = healthy plants)
    # np.random.randn() generates random numbers from normal distribution
    # mean=0, std=1, so 0.6 + 0.15*randn gives values mostly 0.45-0.75
    base_ndvi = 0.6 + 0.15 * np.random.randn(height, width)
    
    # Add a river (water has negative NDVI)
    # np.sin() creates a wavy pattern across the image
    river_y = np.sin(np.linspace(0, 2*np.pi, width)) * 10 + 50
    for x in range(width):
        y = int(river_y[x])  # Convert float to integer index
        if 0 <= y < height:  # Check y is within image bounds
            # Set river pixels (y-2 to y+3) to water NDVI value
            base_ndvi[max(0, y-2):min(height, y+3), x] = -0.2
    
    # Add urban area in corner (low NDVI = buildings, roads)
    # Slice notation [0:25, 0:25] selects top-left 25x25 pixels
    base_ndvi[0:25, 0:25] = 0.1 + 0.05 * np.random.randn(25, 25)
    
    # Clip values to valid NDVI range (-1 to 1)
    # np.clip() limits values to specified min/max
    ndvi_before = np.clip(base_ndvi, -1, 1)
    
    # --------------------------------------------------------
    # CREATE "AFTER" IMAGE (with vegetation clearing)
    # --------------------------------------------------------
    
    # Start with a copy of "before" image
    # .copy() is important! Without it, changes affect both arrays
    ndvi_after = ndvi_before.copy()
    
    # Simulate vegetation clearing in a rectangular area
    # This represents deforestation, development, or fire damage
    clearing_y1, clearing_y2 = 40, 70  # Row range
    clearing_x1, clearing_x2 = 50, 80  # Column range
    
    # Set cleared area to low NDVI (bare soil ~ 0.1-0.2)
    cleared_height = clearing_y2 - clearing_y1
    cleared_width = clearing_x2 - clearing_x1
    ndvi_after[clearing_y1:clearing_y2, clearing_x1:clearing_x2] = (
        0.15 + 0.05 * np.random.randn(cleared_height, cleared_width)
    )
    
    ndvi_after = np.clip(ndvi_after, -1, 1)
    
    # --------------------------------------------------------
    # STORE METADATA FOR LATER USE
    # --------------------------------------------------------
    sample_ndvi = {
        'before': ndvi_before,
        'after': ndvi_after,
        'transform': transform,
        'crs': 'EPSG:4326',
        'bounds': (minx, miny, maxx, maxy)
    }
    
    # Create study area boundary (Area of Interest)
    aoi = gpd.GeoDataFrame(
        {'name': ['Study Area']},
        geometry=[box(minx, miny, maxx, maxy)],
        crs='EPSG:4326'
    )
    
    # Create analysis zones (4 quadrants for zonal statistics)
    mid_x = (minx + maxx) / 2
    mid_y = (miny + maxy) / 2
    zones = gpd.GeoDataFrame({
        'name': ['Northwest', 'Northeast', 'Southwest', 'Southeast'],
        'geometry': [
            box(minx, mid_y, mid_x, maxy),  # NW
            box(mid_x, mid_y, maxx, maxy),  # NE
            box(minx, miny, mid_x, mid_y),  # SW
            box(mid_x, miny, maxx, mid_y)   # SE
        ]
    }, crs='EPSG:4326')
    
    print("Sample data generated:")
    print(f"  Image size: {width} × {height} pixels")
    print(f"  Area: ~10km × 10km (Sydney region)")
    print(f"  Scenario: Forest clearing in southeast quadrant")

---

## Understanding NDVI

### What is NDVI?

**NDVI (Normalized Difference Vegetation Index)** exploits a key property of plants:
- Healthy vegetation **absorbs red light** for photosynthesis
- Healthy vegetation **reflects near-infrared (NIR) light**

### The formula

```
NDVI = (NIR - Red) / (NIR + Red)
```

### Interpreting values

| NDVI | Meaning |
|------|--------|
| 0.6 to 1.0 | Dense, healthy vegetation |
| 0.3 to 0.6 | Moderate vegetation |
| 0.1 to 0.3 | Sparse/stressed vegetation |
| -0.1 to 0.1 | Bare soil, urban areas |
| -1.0 to -0.1 | Water |

### Why normalize?

Dividing by (NIR + Red) removes the effect of overall brightness,
making NDVI comparable across different lighting conditions.

---

## Step 4: Visualize NDVI images

In [ ]:
# ============================================================
# VISUALIZE NDVI BEFORE AND AFTER
# ============================================================

# Get NDVI arrays (from sample data or loaded files)
if USE_SAMPLE_DATA:
    ndvi_before = sample_ndvi['before']
    ndvi_after = sample_ndvi['after']
    raster_transform = sample_ndvi['transform']
else:
    # Load from real files - you'd need to calculate NDVI from bands
    # This is a placeholder for when using real data
    pass

# Create a figure with 2 subplots side by side
# figsize=(width, height) in inches
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot BEFORE image
# imshow() displays 2D arrays as images
# cmap="RdYlGn" = Red-Yellow-Green colormap (red=low, green=high)
# vmin/vmax set the color scale range
im1 = axes[0].imshow(
    ndvi_before,
    cmap="RdYlGn",
    vmin=-0.2,  # Values below this are darkest red
    vmax=0.8    # Values above this are darkest green
)
axes[0].set_title("NDVI - Before", fontsize=12)
axes[0].axis("off")  # Hide axis ticks

# Add colorbar to show what colors mean
# shrink=0.8 makes colorbar 80% of plot height
plt.colorbar(im1, ax=axes[0], shrink=0.8, label="NDVI")

# Plot AFTER image
im2 = axes[1].imshow(ndvi_after, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
axes[1].set_title("NDVI - After", fontsize=12)
axes[1].axis("off")
plt.colorbar(im2, ax=axes[1], shrink=0.8, label="NDVI")

# Add overall title
# y=1.02 positions it slightly above the plots
plt.suptitle("NDVI: Green = healthy vegetation, Red/Yellow = bare/stressed", y=1.02)

# Adjust spacing between subplots
plt.tight_layout()
plt.show()

# Print statistics
print("\nNDVI Statistics:")
print(f"Before - Mean: {np.nanmean(ndvi_before):.3f}, Range: [{np.nanmin(ndvi_before):.3f}, {np.nanmax(ndvi_before):.3f}]")
print(f"After  - Mean: {np.nanmean(ndvi_after):.3f}, Range: [{np.nanmin(ndvi_after):.3f}, {np.nanmax(ndvi_after):.3f}]")

---

## Step 5: Calculate and visualize change

In [ ]:
# ============================================================
# CALCULATE NDVI CHANGE
# ============================================================
# Change detection is simply: After - Before
# Positive = vegetation increase (regrowth)
# Negative = vegetation decrease (clearing, drought, fire)

ndvi_change = ndvi_after - ndvi_before

# Calculate statistics
# np.nanmean() ignores NaN (missing) values
mean_change = np.nanmean(ndvi_change)
min_change = np.nanmin(ndvi_change)
max_change = np.nanmax(ndvi_change)

print("NDVI Change Statistics:")
print(f"  Mean change:  {mean_change:.3f}")
print(f"  Min change:   {min_change:.3f} (biggest loss)")
print(f"  Max change:   {max_change:.3f} (biggest gain)")

# Count pixels with significant change
# A threshold of 0.1 is commonly used for "significant" change
significant_decrease = np.sum(ndvi_change < -0.1)  # Count pixels < -0.1
significant_increase = np.sum(ndvi_change > 0.1)   # Count pixels > 0.1
total_pixels = np.sum(~np.isnan(ndvi_change))      # Total valid pixels

print(f"\nPixels with significant change (threshold = ±0.1):")
print(f"  Vegetation loss:  {significant_decrease:,} pixels ({100*significant_decrease/total_pixels:.1f}%)")
print(f"  Vegetation gain:  {significant_increase:,} pixels ({100*significant_increase/total_pixels:.1f}%)")

In [ ]:
# ============================================================
# VISUALIZE CHANGE WITH THREE-PANEL PLOT
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Before
im0 = axes[0].imshow(ndvi_before, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
axes[0].set_title("NDVI Before")
axes[0].axis("off")
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# Panel 2: After
im1 = axes[1].imshow(ndvi_after, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
axes[1].set_title("NDVI After")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], shrink=0.8)

# Panel 3: Change
# Note: vmin=-0.5, vmax=0.5 centers zero at middle of colormap
# This makes red=loss, white=no change, green=gain
im2 = axes[2].imshow(ndvi_change, cmap="RdYlGn", vmin=-0.5, vmax=0.5)
axes[2].set_title("NDVI Change\n(Red = loss, Green = gain)")
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], shrink=0.8, label="Change")

plt.tight_layout()
plt.show()

if USE_SAMPLE_DATA:
    print("\nThe red rectangle shows the simulated vegetation clearing.")

---

## Step 6: Zonal statistics

Zonal statistics summarize raster values within polygon boundaries.
This is useful for reporting change by administrative area, land parcel, etc.

In [ ]:
# ============================================================
# CALCULATE ZONAL STATISTICS
# ============================================================
# rasterstats library calculates statistics for each polygon zone

from rasterstats import zonal_stats

# zonal_stats() takes:
# - vectors: GeoDataFrame with polygon zones
# - raster: 2D numpy array of values
# - affine: geotransform that positions the raster
# - stats: list of statistics to calculate
# - nodata: value to treat as missing

stats = zonal_stats(
    zones,                          # Our 4 quadrant zones
    ndvi_change,                    # The change raster
    affine=raster_transform,        # Spatial reference
    stats=["mean", "min", "max", "count"],
    nodata=np.nan                   # Treat NaN as missing
)

# stats is a list of dictionaries, one per zone
# Add results to the zones GeoDataFrame
zones["ndvi_change_mean"] = [s["mean"] for s in stats]
zones["ndvi_change_min"] = [s["min"] for s in stats]
zones["ndvi_change_max"] = [s["max"] for s in stats]
zones["pixel_count"] = [s["count"] for s in stats]

print("Zonal Statistics - NDVI Change by Zone:")
print(zones[["name", "ndvi_change_mean", "ndvi_change_min", "ndvi_change_max"]].to_string(index=False))

In [ ]:
# ============================================================
# MAP ZONAL RESULTS
# ============================================================

fig, ax = plt.subplots(figsize=(10, 8))

# Plot zones colored by mean NDVI change
zones.plot(
    column="ndvi_change_mean",
    cmap="RdYlGn",
    legend=True,
    legend_kwds={"label": "Mean NDVI Change"},
    edgecolor="black",
    linewidth=2,
    ax=ax
)

# Add labels showing zone name and mean change value
for idx, row in zones.iterrows():
    centroid = row.geometry.centroid
    ax.annotate(
        f"{row['name']}\n{row['ndvi_change_mean']:.3f}",
        xy=(centroid.x, centroid.y),
        ha='center', va='center',
        fontsize=10, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7)
    )

ax.set_title("Mean NDVI Change by Zone")
ax.set_axis_off()
plt.show()

---

# Part B: Digital Elevation Model (DEM) Analysis

DEMs represent terrain elevation. From elevation data, we can derive:
- **Slope** — Steepness of terrain (in degrees or percent)
- **Aspect** — Direction the slope faces (N, S, E, W)
- **Hillshade** — Simulated illumination for visualization

In [ ]:
# ============================================================
# GENERATE SAMPLE DEM DATA
# ============================================================
# We create a synthetic DEM with realistic terrain features:
# - A mountain peak
# - A valley
# - General rolling terrain

np.random.seed(123)

# DEM dimensions (same as satellite data)
dem_height, dem_width = 100, 100

# Create coordinate grids
# np.meshgrid creates 2D arrays of x and y coordinates
x = np.linspace(0, 10, dem_width)    # 0 to 10 km in x direction
y = np.linspace(0, 10, dem_height)   # 0 to 10 km in y direction
X, Y = np.meshgrid(x, y)

# Base terrain using sine waves (creates rolling hills)
# np.sin() of coordinates creates periodic undulations
base_terrain = (
    50 * np.sin(X * 0.5) * np.cos(Y * 0.3) +  # Large-scale hills
    20 * np.sin(X * 1.2 + Y * 0.8) +          # Medium undulations
    10 * np.random.randn(dem_height, dem_width)  # Small-scale roughness
)

# Add a mountain peak (Gaussian bump)
# Gaussian = bell curve shape, creates smooth peak
peak_x, peak_y = 7, 3  # Peak location
mountain = 200 * np.exp(-((X - peak_x)**2 + (Y - peak_y)**2) / 8)

# Add a valley (negative Gaussian)
valley_x, valley_y = 3, 7
valley = -80 * np.exp(-((X - valley_x)**2 + (Y - valley_y)**2) / 10)

# Combine all features
# Add 500m base elevation (sea level offset)
dem = 500 + base_terrain + mountain + valley

# Ensure no negative elevations
dem = np.maximum(dem, 0)

print("Generated sample DEM:")
print(f"  Size: {dem_width} × {dem_height} pixels")
print(f"  Elevation range: {dem.min():.1f}m to {dem.max():.1f}m")
print(f"  Mean elevation: {dem.mean():.1f}m")

In [ ]:
# ============================================================
# VISUALIZE THE DEM
# ============================================================

fig, ax = plt.subplots(figsize=(10, 8))

# Use 'terrain' colormap: blue=low, green=medium, brown=high, white=peaks
im = ax.imshow(dem, cmap='terrain')
plt.colorbar(im, ax=ax, label='Elevation (m)', shrink=0.8)

ax.set_title('Digital Elevation Model (DEM)', fontsize=14)
ax.set_xlabel('X (pixels)')
ax.set_ylabel('Y (pixels)')

plt.tight_layout()
plt.show()

---

## Step 7: Calculate slope

**Slope** measures the steepness of terrain - how quickly elevation changes.

We calculate slope using the **gradient** (rate of change) in x and y directions,
then combine them using the Pythagorean theorem.

In [ ]:
# ============================================================
# CALCULATE SLOPE
# ============================================================
# Slope = arctan(√(dz/dx)² + (dz/dy)²)
#
# Where:
# - dz/dx = rate of elevation change in x direction
# - dz/dy = rate of elevation change in y direction

# Cell size in meters (for proper gradient calculation)
# Our 100x100 pixel grid covers 10km, so each pixel = 100m
cell_size = 100  # meters

# np.gradient() calculates the rate of change between adjacent cells
# Returns two arrays: gradient in y direction, gradient in x direction
# We multiply by cell_size to get proper units (meters rise per meter run)
dz_dy, dz_dx = np.gradient(dem, cell_size)

# Calculate slope magnitude using Pythagorean theorem
# This gives "rise over run" (dimensionless)
slope_gradient = np.sqrt(dz_dx**2 + dz_dy**2)

# Convert to degrees using arctangent
# np.arctan() returns radians, np.degrees() converts to degrees
slope_degrees = np.degrees(np.arctan(slope_gradient))

print("Slope Statistics:")
print(f"  Min slope:  {slope_degrees.min():.1f}°")
print(f"  Max slope:  {slope_degrees.max():.1f}°")
print(f"  Mean slope: {slope_degrees.mean():.1f}°")
print(f"\nSlope interpretation:")
print(f"  0-5°:   Flat to gentle (easy walking)")
print(f"  5-15°:  Moderate (noticeable incline)")
print(f"  15-30°: Steep (difficult terrain)")
print(f"  >30°:   Very steep (cliffs, avalanche risk)")

In [ ]:
# ============================================================
# VISUALIZE SLOPE
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: DEM for reference
im0 = axes[0].imshow(dem, cmap='terrain')
axes[0].set_title('Elevation (DEM)')
plt.colorbar(im0, ax=axes[0], label='Elevation (m)', shrink=0.8)

# Right: Slope
# Use 'YlOrRd' (Yellow-Orange-Red) where darker = steeper
im1 = axes[1].imshow(slope_degrees, cmap='YlOrRd', vmin=0, vmax=45)
axes[1].set_title('Slope (degrees)')
plt.colorbar(im1, ax=axes[1], label='Slope (°)', shrink=0.8)

for ax in axes:
    ax.set_xlabel('X (pixels)')
    ax.set_ylabel('Y (pixels)')

plt.tight_layout()
plt.show()

print("Notice: Steepest slopes are on the sides of the mountain peak (bottom-right).")

---

## Step 8: Calculate aspect

**Aspect** is the compass direction a slope faces (0°=North, 90°=East, 180°=South, 270°=West).

Aspect affects:
- Solar exposure (south-facing slopes get more sun in Southern Hemisphere)
- Vegetation patterns
- Snow accumulation and melt

In [ ]:
# ============================================================
# CALCULATE ASPECT
# ============================================================
# Aspect = arctan2(dz/dx, dz/dy) converted to compass bearing
#
# np.arctan2(y, x) returns angle in radians from -π to π
# We convert to compass degrees (0-360, clockwise from North)

# Calculate aspect using arctan2
# Note: we use -dz_dx, -dz_dy because aspect is the direction
# the slope FACES (downhill direction), not uphill
aspect_radians = np.arctan2(-dz_dx, -dz_dy)

# Convert from radians to degrees
aspect_degrees = np.degrees(aspect_radians)

# Convert from (-180 to 180) to (0 to 360) compass bearing
# Negative angles become positive by adding 360
aspect_degrees = np.where(aspect_degrees < 0, aspect_degrees + 360, aspect_degrees)

print("Aspect Statistics:")
print(f"  Range: {aspect_degrees.min():.1f}° to {aspect_degrees.max():.1f}°")
print(f"\nAspect interpretation (compass direction slope faces):")
print(f"  0° / 360°: North-facing")
print(f"  90°:       East-facing")
print(f"  180°:      South-facing")
print(f"  270°:      West-facing")

In [ ]:
# ============================================================
# VISUALIZE ASPECT
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: DEM for reference
im0 = axes[0].imshow(dem, cmap='terrain')
axes[0].set_title('Elevation (DEM)')
plt.colorbar(im0, ax=axes[0], label='Elevation (m)', shrink=0.8)

# Right: Aspect with circular colormap
# 'hsv' colormap is circular: both ends are the same color (red)
# This is perfect for aspect since 0° and 360° are both North
im1 = axes[1].imshow(aspect_degrees, cmap='hsv', vmin=0, vmax=360)
axes[1].set_title('Aspect (compass direction)')
cbar = plt.colorbar(im1, ax=axes[1], label='Aspect (°)', shrink=0.8)
cbar.set_ticks([0, 90, 180, 270, 360])
cbar.set_ticklabels(['N (0°)', 'E (90°)', 'S (180°)', 'W (270°)', 'N (360°)'])

for ax in axes:
    ax.set_xlabel('X (pixels)')
    ax.set_ylabel('Y (pixels)')

plt.tight_layout()
plt.show()

print("The circular colormap shows slope direction - each color is a compass bearing.")

---

## Step 9: Calculate hillshade

**Hillshade** simulates how terrain would look with sunlight from a specific direction.
It's purely for visualization - it makes terrain features much easier to see.

In [ ]:
# ============================================================
# CALCULATE HILLSHADE
# ============================================================
# Hillshade simulates illumination from a light source.
#
# Parameters:
# - azimuth: Direction of light source (315° = northwest, typical)
# - altitude: Height of sun above horizon (45° = typical)
#
# Formula uses dot product of surface normal and light direction.

def calculate_hillshade(elevation, cell_size, azimuth=315, altitude=45):
    """
    Calculate hillshade from a DEM.
    
    Parameters:
    -----------
    elevation : 2D array
        Elevation values in meters
    cell_size : float
        Size of each cell in meters
    azimuth : float
        Direction of light source in degrees (0=N, 90=E, 180=S, 270=W)
        Default 315 = northwest (standard cartographic convention)
    altitude : float
        Height of light source above horizon in degrees
        Default 45 = mid-height sun
    
    Returns:
    --------
    hillshade : 2D array
        Values from 0 (shadow) to 255 (full illumination)
    """
    # Convert angles to radians
    azimuth_rad = np.radians(360 - azimuth + 90)  # Convert to math convention
    altitude_rad = np.radians(altitude)
    
    # Calculate gradients
    dz_dy, dz_dx = np.gradient(elevation, cell_size)
    
    # Calculate slope and aspect
    slope = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))
    aspect = np.arctan2(-dz_dx, dz_dy)
    
    # Calculate hillshade
    # This is the dot product of the surface normal and light direction
    hillshade = (
        np.sin(altitude_rad) * np.cos(slope) +
        np.cos(altitude_rad) * np.sin(slope) * np.cos(azimuth_rad - aspect)
    )
    
    # Scale to 0-255 range
    hillshade = np.clip(hillshade, 0, 1) * 255
    
    return hillshade

# Calculate hillshade with default parameters
hillshade = calculate_hillshade(dem, cell_size=100)

print("Hillshade calculated.")
print(f"  Light direction: 315° (northwest)")
print(f"  Light altitude: 45° above horizon")
print(f"  Value range: {hillshade.min():.0f} to {hillshade.max():.0f}")

In [ ]:
# ============================================================
# VISUALIZE HILLSHADE
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Raw DEM
im0 = axes[0].imshow(dem, cmap='terrain')
axes[0].set_title('DEM (Elevation)')
plt.colorbar(im0, ax=axes[0], shrink=0.8, label='m')

# Panel 2: Hillshade only
# Use grayscale colormap for illumination
im1 = axes[1].imshow(hillshade, cmap='gray')
axes[1].set_title('Hillshade\n(simulated illumination)')

# Panel 3: DEM + Hillshade combined
# This creates a nice 3D-like visualization
im2 = axes[2].imshow(dem, cmap='terrain')
# Overlay hillshade with transparency (alpha)
axes[2].imshow(hillshade, cmap='gray', alpha=0.5)
axes[2].set_title('DEM + Hillshade\n(combined for depth)')
plt.colorbar(im2, ax=axes[2], shrink=0.8, label='m')

for ax in axes:
    ax.axis('off')

plt.tight_layout()
plt.show()

print("The combined view (right) is how professional maps show terrain.")

---

## Step 10: Complete terrain analysis view

In [ ]:
# ============================================================
# COMPLETE TERRAIN ANALYSIS - 4 PANEL VIEW
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Top-left: DEM with hillshade
axes[0, 0].imshow(dem, cmap='terrain')
axes[0, 0].imshow(hillshade, cmap='gray', alpha=0.4)
axes[0, 0].set_title('Elevation with Hillshade', fontsize=12)

# Top-right: Slope
im_slope = axes[0, 1].imshow(slope_degrees, cmap='YlOrRd', vmin=0, vmax=40)
axes[0, 1].set_title('Slope (degrees)', fontsize=12)
plt.colorbar(im_slope, ax=axes[0, 1], shrink=0.8, label='°')

# Bottom-left: Aspect
im_aspect = axes[1, 0].imshow(aspect_degrees, cmap='hsv', vmin=0, vmax=360)
axes[1, 0].set_title('Aspect (compass direction)', fontsize=12)
cbar = plt.colorbar(im_aspect, ax=axes[1, 0], shrink=0.8)
cbar.set_ticks([0, 90, 180, 270, 360])
cbar.set_ticklabels(['N', 'E', 'S', 'W', 'N'])

# Bottom-right: Slope classification
# Create classified slope map
slope_classes = np.zeros_like(slope_degrees)
slope_classes[(slope_degrees >= 0) & (slope_degrees < 5)] = 1    # Flat
slope_classes[(slope_degrees >= 5) & (slope_degrees < 15)] = 2   # Moderate
slope_classes[(slope_degrees >= 15) & (slope_degrees < 30)] = 3  # Steep
slope_classes[slope_degrees >= 30] = 4                            # Very steep

# Custom colormap for classes
cmap_classes = colors.ListedColormap(['white', 'green', 'yellow', 'orange', 'red'])
im_class = axes[1, 1].imshow(slope_classes, cmap=cmap_classes, vmin=0, vmax=4)
axes[1, 1].set_title('Slope Classification', fontsize=12)
cbar_class = plt.colorbar(im_class, ax=axes[1, 1], shrink=0.8)
cbar_class.set_ticks([0.5, 1.5, 2.5, 3.5])
cbar_class.set_ticklabels(['Flat\n<5°', 'Moderate\n5-15°', 'Steep\n15-30°', 'Very Steep\n>30°'])

for ax in axes.flat:
    ax.axis('off')

plt.suptitle('Complete Terrain Analysis', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## Step 11: Export results

In [ ]:
# ============================================================
# EXPORT ANALYSIS RESULTS
# ============================================================

# Save NDVI zones with statistics as GeoPackage
zones.to_file(PROCESSED / "ndvi_zones.gpkg", driver="GPKG")
print(f"Saved NDVI zones to: {PROCESSED / 'ndvi_zones.gpkg'}")

# Save terrain analysis as GeoTIFFs
# We'll save slope and hillshade as raster files

def save_raster(data, path, transform, crs='EPSG:4326'):
    """Save a 2D numpy array as a GeoTIFF."""
    with rasterio.open(
        path,
        'w',
        driver='GTiff',
        height=data.shape[0],
        width=data.shape[1],
        count=1,
        dtype=data.dtype,
        crs=crs,
        transform=transform
    ) as dst:
        dst.write(data, 1)

# Save slope
save_raster(
    slope_degrees.astype('float32'),
    PROCESSED / "slope_degrees.tif",
    raster_transform
)
print(f"Saved slope to: {PROCESSED / 'slope_degrees.tif'}")

# Save hillshade
save_raster(
    hillshade.astype('float32'),
    PROCESSED / "hillshade.tif",
    raster_transform
)
print(f"Saved hillshade to: {PROCESSED / 'hillshade.tif'}")

# Save NDVI change
save_raster(
    ndvi_change.astype('float32'),
    PROCESSED / "ndvi_change.tif",
    raster_transform
)
print(f"Saved NDVI change to: {PROCESSED / 'ndvi_change.tif'}")

print("\nAll files saved! You can open these in QGIS.")

---

## Summary

### Part A: Satellite Imagery
- **NDVI** measures vegetation health using NIR and Red bands
- **Change detection** compares NDVI between dates
- **Zonal statistics** summarizes change by area

### Part B: Terrain Analysis
- **Slope** = steepness (calculated from elevation gradients)
- **Aspect** = compass direction the slope faces
- **Hillshade** = simulated illumination for visualization

### Key numpy functions used
| Function | What it does |
|----------|-------------|
| `np.gradient()` | Calculate rate of change between cells |
| `np.arctan()` | Inverse tangent (for slope) |
| `np.arctan2()` | Inverse tangent preserving quadrant (for aspect) |
| `np.degrees()` | Convert radians to degrees |
| `np.clip()` | Limit values to a range |
| `np.where()` | Conditional selection |

---

## Using real data

### Satellite imagery
See the detailed Copernicus download instructions in the [data download guide](../onboarding/03-download-data.md).

### DEM data
- **Australia**: [ELVIS](https://elevation.fsdf.org.au/)
- **Global**: [USGS EarthExplorer](https://earthexplorer.usgs.gov/) - search for SRTM
- **Cloud**: [Microsoft Planetary Computer](https://planetarycomputer.microsoft.com/) - Copernicus DEM

---

**Save your work:**
- Colab: `File > Save a copy in Drive`
- Local: `Ctrl+S` or `Cmd+S`